In [3]:
import os, random
import pandas as pd

from torch.utils.data import Subset
import numpy as np

from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from PIL import ImageFile, UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.models import resnet50, ResNet50_Weights

INPUT_DIR = "/kaggle/input/datasets/shashiprabhapanwar/wikiart"
IMAGES_ROOT = os.path.join(INPUT_DIR, "wikiart_clean", "wikiart")
TRAIN_CSV = os.path.join(INPUT_DIR, "style_train_clean.csv")
VAL_CSV   = os.path.join(INPUT_DIR, "style_val_lite_clean.csv")

In [4]:
def read_wikiart_csv(path):
    df = pd.read_csv(path)

    # headerless got misread as header (common)
    if len(df.columns) == 2:
        c0, c1 = df.columns[0], df.columns[1]
        if ("/" in str(c0)) and str(c1).strip().isdigit():
            df = pd.read_csv(path, header=None, names=["relpath", "label"])
            df["label"] = df["label"].astype(int)
            return df

    # normal header (strip spaces)
    df.columns = [c.strip() for c in df.columns]

    # guess columns
    path_col = df.columns[0]
    label_col = df.columns[1]
    for c in df.columns:
        lc = c.lower()
        if lc in ["pictures", "picture", "path", "file", "filename", "image"]:
            path_col = c
        if lc in ["class", "label", "target"]:
            label_col = c

    df = df[[path_col, label_col]].rename(columns={path_col:"relpath", label_col:"label"})
    df["label"] = df["label"].astype(int)
    return df


def filter_bad_images(df, images_root):
    good_rows = []
    bad = 0

    for relpath, label in zip(df["relpath"].astype(str).values, df["label"].values):
        fullpath = os.path.join(images_root, relpath)
        try:
            with Image.open(fullpath) as im:
                im.verify()  # verifies file integrity without decoding full image
            good_rows.append((relpath, int(label)))
        except (OSError, UnidentifiedImageError):
            bad += 1

    out = pd.DataFrame(good_rows, columns=["relpath", "label"])
    print(f"Filtered bad images: {bad} removed, {len(out)} remaining")
    return out

In [5]:
class WikiArtDataset(Dataset):
    def __init__(self, df, images_root, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.images_root = images_root
        self.transform = transform

        self.df["fullpath"] = self.df["relpath"].astype(str).apply(lambda p: os.path.join(images_root, p))
        # (you already verified 0 missing, so no filtering here)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["fullpath"]
        label = int(row["label"])
    
        img = Image.open(img_path)
        img = img.convert("RGB") 
        if self.transform:
            img = self.transform(img)
        return img, label

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class AttnPool1D(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)

    def forward(self, x):
        # x: (B, T, D)
        a = self.w(x).squeeze(-1)              # (B, T)
        a = torch.softmax(a, dim=1).unsqueeze(-1)  # (B, T, 1)
        return (x * a).sum(dim=1)              # (B, D)

class ResNet50_FusionCRNN(nn.Module):
    def __init__(self, num_classes, rnn_in=256, rnn_hidden=128, dropout=0.3):
        super().__init__()
        base = resnet50(weights=ResNet50_Weights.DEFAULT)

        # keep resnet structure but split it
        self.stem = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4
        )

        # baseline GAP branch
        self.gap = nn.AdaptiveAvgPool2d(1)

        # CRNN branch (lightweight)
        self.proj = nn.Sequential(
            nn.Conv2d(2048, rnn_in, kernel_size=1, bias=False),
            nn.BatchNorm2d(rnn_in),
            nn.ReLU(inplace=True),
        )
        self.rnn = nn.GRU(
            input_size=rnn_in,
            hidden_size=rnn_hidden,
            num_layers=1,           # back to 1 — simpler, less overfitting risk
            batch_first=True,
            bidirectional=True,
        )
        self.pool = AttnPool1D(2 * rnn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(2048 + 2 * rnn_hidden, num_classes)

    def forward(self, x):
        f = self.stem(x)  # (B,2048,7,7)

        # GAP baseline vector
        gap = self.gap(f).flatten(1)  # (B,2048)

        z = self.proj(f)              # (B, rnn_in, 7, 7)
        seq = z.flatten(2)            # (B, rnn_in, 49)
        seq = seq.permute(0, 2, 1)   # (B, 49, rnn_in)

        out, _ = self.rnn(seq)        # (B,7,2*rnn_hidden)
        rnn_vec = self.pool(out)      # (B,2*rnn_hidden)

        fused = torch.cat([gap, rnn_vec], dim=1)
        fused = self.dropout(fused)
        return self.head(fused)

In [6]:
def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

def make_optimizer_and_scheduler(model, lr_head=1e-3, lr_backbone=5e-5, wd=1e-4):
    # params for CRNN head (always train these)
    head_params = []
    for name in ["proj", "rnn", "pool", "head", "dropout"]:
        if hasattr(model, name):
            head_params += list(getattr(model, name).parameters())

    # backbone params (stem)
    backbone_params = list(model.stem.parameters())

    # Only include params that are currently trainable
    param_groups = []
    head_params = [p for p in head_params if p.requires_grad]
    backbone_params = [p for p in backbone_params if p.requires_grad]

    if head_params:
        param_groups.append({"params": head_params, "lr": lr_head, "weight_decay": wd})
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": lr_backbone, "weight_decay": wd})

    optimizer = torch.optim.AdamW(param_groups)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.3)
    return optimizer, scheduler

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_df = read_wikiart_csv(TRAIN_CSV)
SUBSET_N = 10000
train_df = train_df.sample(n=min(SUBSET_N, len(train_df)), random_state=42).reset_index(drop=True)
train_df = filter_bad_images(train_df, IMAGES_ROOT)

val_df = read_wikiart_csv(VAL_CSV)

num_classes = int(max(train_df["label"].max(), val_df["label"].max())) + 1
print("num_classes:", num_classes)
print("train/val sizes:", len(train_df), len(val_df))

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = WikiArtDataset(train_df, IMAGES_ROOT, transform=train_transform)
val_dataset = WikiArtDataset(val_df, IMAGES_ROOT, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False,
                        num_workers=4, pin_memory=True, persistent_workers=True)

# ─── Model ───
model = ResNet50_FusionCRNN(num_classes=num_classes, rnn_in=256, rnn_hidden=256, dropout=0.35).to(device)

for i in range(4):
    set_requires_grad(model.stem[i], False)
for i in range(4, 8):
    set_requires_grad(model.stem[i], True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

backbone_params = []
for i in range(4, 8):
    backbone_params += list(model.stem[i].parameters())

head_params = []
for name in ["proj", "rnn", "pool", "head"]:
    head_params += list(getattr(model, name).parameters())

optimizer = optim.AdamW([
    {"params": head_params, "lr": 1e-3, "weight_decay": 1e-4},
    {"params": [p for p in backbone_params if p.requires_grad], "lr": 3e-4, "weight_decay": 1e-4},
])

def topk_acc(logits, y, k=5):
    k = min(k, logits.size(1))
    _, topk = logits.topk(k, dim=1)
    return (topk == y.unsqueeze(1)).any(dim=1).float().mean().item()

# Mixup, used for mixing up two pic with different classes like x% class A and y% class B to train model on soft probablity 
def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

epochs=20

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for epoch in range(1, epochs + 1):
    model.train()
    loss_sum = 0.0
    steps = 0

    for x, y in train_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        x_mix, y_a, y_b, lam = mixup_data(x, y, alpha=0.2)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits = model(x_mix)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item()
        steps += 1

    scheduler.step()

    model.eval()
    correct1 = 0
    total = 0
    top5_sum = 0.0
    batches = 0

    with torch.no_grad():
        for x, y in val_loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x)

            pred = logits.argmax(dim=1)
            total += y.size(0)
            correct1 += (pred == y).sum().item()

            top5_sum += topk_acc(logits, y, k=5)
            batches += 1

    val_top1 = 100.0 * correct1 / max(1, total)
    val_top5 = 100.0 * (top5_sum / max(1, batches))

    lrs = [pg["lr"] for pg in optimizer.param_groups]
    print(f"Epoch {epoch:02d}/{epochs} | lrs {lrs} | "
          f"train_loss {loss_sum/max(1,steps):.4f} | "
          f"val_top1 {val_top1:.2f}% | val_top5 {val_top5:.2f}%")

Device: cuda
Filtered bad images: 0 removed, 10000 remaining
num_classes: 27
train/val sizes: 10000 10178
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s]


Epoch 01/20 | lrs [0.001, 0.0003] | train_loss 2.3755 | val_top1 41.01% | val_top5 83.24%
Epoch 02/20 | lrs [0.001, 0.0003] | train_loss 2.0640 | val_top1 45.79% | val_top5 85.51%
Epoch 03/20 | lrs [0.001, 0.0003] | train_loss 1.8076 | val_top1 47.97% | val_top5 86.97%
Epoch 04/20 | lrs [0.0005, 0.00015] | train_loss 1.6787 | val_top1 48.57% | val_top5 87.05%
Epoch 05/20 | lrs [0.0005, 0.00015] | train_loss 1.4602 | val_top1 51.50% | val_top5 88.58%
Epoch 06/20 | lrs [0.0005, 0.00015] | train_loss 1.3130 | val_top1 52.53% | val_top5 88.61%
Epoch 07/20 | lrs [0.0005, 0.00015] | train_loss 1.3451 | val_top1 51.79% | val_top5 87.84%
Epoch 08/20 | lrs [0.00025, 7.5e-05] | train_loss 1.1995 | val_top1 51.95% | val_top5 87.30%
Epoch 09/20 | lrs [0.00025, 7.5e-05] | train_loss 1.1387 | val_top1 52.03% | val_top5 87.56%
Epoch 10/20 | lrs [0.00025, 7.5e-05] | train_loss 1.1645 | val_top1 52.19% | val_top5 86.64%
Epoch 11/20 | lrs [0.00025, 7.5e-05] | train_loss 1.1255 | val_top1 51.97% | val_to

# 